# ☀️ Unified SOTA Benchmark: Cross-Paper Model Inference on Single Dataset

This notebook provides an **apples-to-apples, publication-grade benchmark** by running inference for **multiple published research models** on the **exact same frozen Google satellite test dataset** on an NVIDIA T4 GPU.

### 🔬 Evaluated Models & Literature Checkpoints:
1. **Kleebauer et al. (2023)**: DeepLabV3+ (ResNet-101) — Fraunhofer ISE
2. **Moreno et al. (2023)**: YOLOv8-seg (ProtoNet) — Univ. Politécnica de Madrid
3. **Moreno et al. (2023)**: UNet++ (ResNet-34) — Univ. Politécnica de Madrid
4. **Microsoft AI for Good (2023)**: Global Renewables Watch (U-Net + ResNeXt-50)
5. **Moreno et al. (2023)**: Baseline SegFormer (MiT-B0) — Univ. Politécnica de Madrid
6. **Our Proposed Model**: Multi-Scale SegFormer (Mix Transformer MiT-B2) with Compound Boundary Loss

### 🌍 Single Frozen Test Dataset:
- **BDAPPV Google Satellite Test Split** (*Nature Scientific Data*, 2023; Kasmi et al.):
  - Authentic Google Earth Engine satellite imagery ($0.1\text{–}0.2\,\text{m/pixel}$).
  - Contains both **solar-positive arrays** AND **hard-negative empty roofs/roads** to evaluate false-alarm suppression.

## 1. Setup & Environment Verification

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q segmentation-models-pytorch datasets ultralytics rasterio geopandas matplotlib seaborn albumentations tabulate timm opencv-python pyarrow

In [ ]:
import os
import time
import urllib.request
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm
from tabulate import tabulate
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import segmentation_models_pytorch as smp
from ultralytics import YOLO
from datasets import load_dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 PyTorch Device: {device}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB VRAM)")

## 2. Automated Download of Literature Pretrained Weights

In [ ]:
weights_dir = "pretrained_models"
os.makedirs(weights_dir, exist_ok=True)

weight_urls = {
    # 1. Kleebauer et al. (2023) - Fraunhofer ISE
    "kleebaue_deeplabv3.pt": "https://owncloud.fraunhofer.de/index.php/s/8MRTJkxmCNA7vin/download",
    # 2. Juan Moreno (2023) - YOLOv8-seg Google Satellite
    "moreno_yolov8_google.pt": "https://raw.githubusercontent.com/juanmooreno/Solar-Panel-Segmentation-DL/main/weights/Google%20Dataset/best_yolov8_google.pt",
    # 3. Juan Moreno (2023) - UNet++ Google Satellite
    "moreno_unet_plusplus.pth": "https://raw.githubusercontent.com/juanmooreno/Solar-Panel-Segmentation-DL/main/weights/Google%20Dataset/best_unet_plusplus_google.pth",
    # 4. Microsoft Global Renewables Watch - U-Net (ResNeXt-50)
    "microsoft_grw_unet.ckpt": "https://github.com/microsoft/global-renewables-watch/releases/download/v1.1/solar_model.ckpt",
    # 5. Juan Moreno (2023) - Baseline SegFormer (MiT-B0)
    "moreno_segformer_google.pth": "https://raw.githubusercontent.com/juanmooreno/Solar-Panel-Segmentation-DL/main/weights/Google%20Dataset/best_segformer_google.pth",
    # 6. Our Proposed Model - Multi-Scale SegFormer (MiT-B2) on Hugging Face
    "best_segformer_large.pth": "https://huggingface.co/abhinav03700/solar-panel-segformer-mit-b2/resolve/main/best_segformer_large.pth"
}

for fname, url in weight_urls.items():
    dst = os.path.join(weights_dir, fname)
    if not os.path.exists(dst):
        print(f"📥 Downloading {fname}...")
        try:
            urllib.request.urlretrieve(url, dst)
            print(f"  ✅ Saved {fname} ({os.path.getsize(dst)/1e6:.1f} MB)")
        except Exception as e:
            print(f"  ⚠️ Error downloading {fname}: {e}")
    else:
        print(f"  ⚡ {fname} cached ({os.path.getsize(dst)/1e6:.1f} MB)")

## 3. Ingesting the Single Frozen Test Dataset (Google Satellite BDAPPV)
We pull 400 test images from the official test split of BDAPPV (`split='test'`), standardizing each to $512\times 512$ with binary `{0, 1}` ground truth masks.

In [ ]:
print("📥 Loading Frozen Test Split from gabrielkasmi/bdappv (Google Satellite)...")
# Slices the official test split
ds_test = load_dataset("gabrielkasmi/bdappv", "google", split="test[:400]")

test_images = []
test_masks = []
pos_count = 0
neg_count = 0

for sample in tqdm(ds_test, desc="Ingesting Test Images"):
    img_arr = np.array(sample["image"].convert("RGB"))
    if sample["has_mask"] and sample["mask"] is not None:
        m_arr = np.array(sample["mask"].convert("L"))
        mask_bin = (m_arr > 128).astype(np.uint8)
        if mask_bin.max() > 0:
            pos_count += 1
        else:
            neg_count += 1
    else:
        mask_bin = np.zeros((img_arr.shape[0], img_arr.shape[1]), dtype=np.uint8)
        neg_count += 1
        
    img_512 = cv2.resize(img_arr, (512, 512), interpolation=cv2.INTER_LINEAR)
    mask_512 = cv2.resize(mask_bin, (512, 512), interpolation=cv2.INTER_NEAREST)
    
    test_images.append(img_512)
    test_masks.append(mask_512)

test_images = np.array(test_images)
test_masks = np.array(test_masks)
print(f"✅ Frozen Test Set Loaded: {len(test_images)} Images ({pos_count} Solar-Positive, {neg_count} Hard-Negatives)!")

## 4. Standardized Model Predictor Wrappers
We encapsulate each architecture behind a uniform API: `predict(image_rgb_512) -> binary_mask_512`.

In [ ]:
models = {}

# -------------------------------------------------------------------------
# 1. Kleebauer et al. (2023) DeepLabV3+ (ResNet-101)
# -------------------------------------------------------------------------
kleebauer_path = os.path.join(weights_dir, "kleebaue_deeplabv3.pt")
if os.path.exists(kleebauer_path):
    try:
        raw_k = torch.load(kleebauer_path, map_location=device, weights_only=False)
    except Exception:
        raw_k = torch.load(kleebauer_path, map_location=device)
    raw_k = raw_k.to(device).eval()
    
    def predict_kleebauer(img_rgb):
        tensor = torch.from_numpy(img_rgb.transpose(2, 0, 1)).float() / 255.0
        tensor = tensor.unsqueeze(0).to(device)
        with torch.no_grad():
            out = raw_k(tensor)
            logits = out["out"] if isinstance(out, dict) else out
            if logits.shape[1] == 1:
                pred = (torch.sigmoid(logits) >= 0.5).squeeze().cpu().numpy().astype(np.uint8)
            else:
                pred = logits.argmax(dim=1).squeeze().cpu().numpy().astype(np.uint8)
        return pred
        
    models["Kleebauer et al. (DeepLabV3+)"] = predict_kleebauer
    print("✅ Loaded Kleebauer et al. DeepLabV3+")

# -------------------------------------------------------------------------
# 2. Juan Moreno (2023) YOLOv8-seg
# -------------------------------------------------------------------------
yolo_path = os.path.join(weights_dir, "moreno_yolov8_google.pt")
if os.path.exists(yolo_path):
    yolo_model = YOLO(yolo_path)
    
    def predict_yolo(img_rgb):
        h, w, _ = img_rgb.shape
        pred = np.zeros((h, w), dtype=np.uint8)
        res = yolo_model.predict(img_rgb, verbose=False, device=0 if torch.cuda.is_available() else 'cpu')[0]
        if res.masks is not None:
            for m in res.masks.data.cpu().numpy():
                m_res = cv2.resize(m, (w, h))
                pred = np.maximum(pred, (m_res > 0.5).astype(np.uint8))
        return pred
        
    models["Moreno et al. (YOLOv8-seg)"] = predict_yolo
    print("✅ Loaded Moreno et al. YOLOv8-seg")

# -------------------------------------------------------------------------
# 3. Juan Moreno (2023) UNet++
# -------------------------------------------------------------------------
unetpp_path = os.path.join(weights_dir, "moreno_unet_plusplus.pth")
if os.path.exists(unetpp_path):
    unetpp_m = smp.UnetPlusPlus(encoder_name="resnet34", in_channels=3, classes=1, activation=None).to(device)
    try:
        unetpp_m.load_state_dict(torch.load(unetpp_path, map_location=device))
        unetpp_m.eval()
        
        def predict_unetpp(img_rgb):
            tensor = torch.from_numpy(img_rgb.transpose(2, 0, 1)).float() / 255.0
            tensor = tensor.unsqueeze(0).to(device)
            with torch.no_grad():
                logits = unetpp_m(tensor)
                pred = (torch.sigmoid(logits) >= 0.5).squeeze().cpu().numpy().astype(np.uint8)
            return pred
            
        models["Moreno et al. (UNet++)"] = predict_unetpp
        print("✅ Loaded Moreno et al. UNet++")
    except Exception as e:
        print(f"⚠️ Could not load UNet++ state dict directly: {e}")

# -------------------------------------------------------------------------
# 4. Microsoft Global Renewables Watch U-Net (ResNeXt-50)
# -------------------------------------------------------------------------
grw_path = os.path.join(weights_dir, "microsoft_grw_unet.ckpt")
if os.path.exists(grw_path):
    try:
        grw_model = smp.Unet(encoder_name="resnext50_32x4d", in_channels=3, classes=2).to(device)
        ckpt = torch.load(grw_path, map_location=device)
        s_dict = ckpt.get("state_dict", ckpt)
        m_weights = {k.replace("model.", ""): v for k, v in s_dict.items() if k.startswith("model.")}
        grw_model.load_state_dict(m_weights if m_weights else s_dict, strict=False)
        grw_model.eval()
        
        def predict_grw(img_rgb):
            tensor = torch.from_numpy(img_rgb.transpose(2, 0, 1)).float() / 255.0
            tensor = tensor.unsqueeze(0).to(device)
            with torch.no_grad():
                logits = grw_model(tensor)
                pred = logits.argmax(dim=1).squeeze(0).cpu().numpy().astype(np.uint8)
            return pred
            
        models["Microsoft (GRW U-Net)"] = predict_grw
        print("✅ Loaded Microsoft Global Renewables Watch U-Net")
    except Exception as e:
        print(f"⚠️ Could not load Microsoft GRW: {e}")

# -------------------------------------------------------------------------
# 5. Juan Moreno (2023) Baseline SegFormer (MiT-B0)
# -------------------------------------------------------------------------
moreno_seg_path = os.path.join(weights_dir, "moreno_segformer_google.pth")
if os.path.exists(moreno_seg_path):
    try:
        base_seg = smp.Segformer(encoder_name="mit_b0", in_channels=3, classes=1, activation=None).to(device)
        base_seg.load_state_dict(torch.load(moreno_seg_path, map_location=device))
        base_seg.eval()
        
        def predict_moreno_seg(img_rgb):
            tensor = torch.from_numpy(img_rgb.transpose(2, 0, 1)).float() / 255.0
            tensor = tensor.unsqueeze(0).to(device)
            with torch.no_grad():
                logits = base_seg(tensor)
                pred = (torch.sigmoid(logits) >= 0.5).squeeze().cpu().numpy().astype(np.uint8)
            return pred
            
        models["Moreno et al. (Baseline SegFormer)"] = predict_moreno_seg
        print("✅ Loaded Moreno et al. Baseline SegFormer (MiT-B0)")
    except Exception as e:
        print(f"⚠️ Could not load Moreno SegFormer: {e}")

# -------------------------------------------------------------------------
# 6. Our Proposed Model: Multi-Scale SegFormer (MiT-B2)
# -------------------------------------------------------------------------
our_segformer = smp.Segformer(encoder_name="mit_b2", in_channels=3, classes=1, activation=None).to(device)
segformer_ckpt = os.path.join(weights_dir, "best_segformer_large.pth")
if not os.path.exists(segformer_ckpt):
    segformer_ckpt = "best_segformer_large.pth"
if not os.path.exists(segformer_ckpt) and os.path.exists("best_segformer_multidataset.pth"):
    segformer_ckpt = "best_segformer_multidataset.pth"

if os.path.exists(segformer_ckpt):
    our_segformer.load_state_dict(torch.load(segformer_ckpt, map_location=device))
    print(f"✅ Loaded Our SegFormer Checkpoint: {segformer_ckpt}")
else:
    print("ℹ️ Initialized SegFormer with ImageNet weights for zero-shot testing.")
our_segformer.eval()

def predict_ours(img_rgb):
    tensor = torch.from_numpy(img_rgb.transpose(2, 0, 1)).float() / 255.0
    tensor = tensor.unsqueeze(0).to(device)
    with torch.no_grad():
        logits = our_segformer(tensor)
        pred = (torch.sigmoid(logits) >= 0.5).squeeze().cpu().numpy().astype(np.uint8)
    return pred

models["Ours: SegFormer (MiT-B2)"] = predict_ours
print(f"\n🎉 Total Runnable Models Configured: {len(models)}")

## 5. Master Benchmark Evaluation Loop across Single Test Set

In [ ]:
benchmark_results = []
predictions_cache = {m_name: [] for m_name in models.keys()}

for model_name, pred_fn in models.items():
    print(f"🚀 Evaluating {model_name} on {len(test_images)} frozen test images...")
    total_tp = 0
    total_fp = 0
    total_fn = 0
    total_tn = 0
    
    # Warmup GPU
    _ = pred_fn(test_images[0])
    
    t_start = time.time()
    for i in tqdm(range(len(test_images)), desc=f"Running {model_name}"):
        img = test_images[i]
        gt = test_masks[i]
        
        pred = pred_fn(img)
        if i < 4:  # Cache first 4 for visual grid
            predictions_cache[model_name].append(pred)
            
        total_tp += np.sum((pred == 1) & (gt == 1))
        total_fp += np.sum((pred == 1) & (gt == 0))
        total_fn += np.sum((pred == 0) & (gt == 1))
        total_tn += np.sum((pred == 0) & (gt == 0))
        
    elapsed = time.time() - t_start
    latency_ms = (elapsed / len(test_images)) * 1000
    fps = 1000.0 / latency_ms
    
    tot_pix = total_tp + total_fp + total_fn + total_tn
    pixel_acc = (total_tp + total_tn) / max(1, tot_pix) * 100
    prec = total_tp / (total_tp + total_fp + 1e-6) * 100
    rec = total_tp / (total_tp + total_fn + 1e-6) * 100
    f1 = 2 * (prec/100) * (rec/100) / ((prec/100) + (rec/100) + 1e-6)
    miou = total_tp / (total_tp + total_fp + total_fn + 1e-6) * 100
    
    benchmark_results.append([
        model_name,
        f"{pixel_acc:.2f} %",
        f"{miou:.2f} %",
        f"{f1:.4f}",
        f"{prec:.2f} %",
        f"{rec:.2f} %",
        f"{latency_ms:.2f} ms",
        f"{fps:.1f} FPS"
    ])

headers = ["Model / Literature Source", "Pixel Acc", "mIoU", "Dice / F1", "Precision", "Recall", "Latency", "Throughput"]
print("\n" + "="*75)
print("🎯 MASTER SOTA BENCHMARK TABLE (EVALUATED ON SINGLE FROZEN TEST SET)")
print("="*75)
print(tabulate(benchmark_results, headers=headers, tablefmt="fancy_grid"))

## 6. Publication-Grade Multi-Model Visual Comparison Grid
Generates a high-contrast side-by-side figure showing 4 test cases across all evaluated models.

In [ ]:
n_samples = 4
n_cols = 2 + len(models)  # Raw, GT, + each model
fig, axes = plt.subplots(n_samples, n_cols, figsize=(3.5 * n_cols, 3.8 * n_samples))

col_titles = ["Raw Satellite", "Ground Truth"] + list(models.keys())

for r in range(n_samples):
    raw = test_images[r]
    gt = test_masks[r]
    
    # 1. Raw
    axes[r, 0].imshow(raw)
    if r == 0: axes[r, 0].set_title("Raw Satellite", fontsize=11, fontweight='bold')
    axes[r, 0].axis('off')
    
    # 2. GT Overlay
    ov_gt = raw.copy()
    ov_gt[gt == 1] = [255, 40, 40]  # Red
    axes[r, 1].imshow(ov_gt)
    if r == 0: axes[r, 1].set_title("Ground Truth", fontsize=11, fontweight='bold')
    axes[r, 1].axis('off')
    
    # 3+ Model predictions
    for m_idx, (m_name, preds_list) in enumerate(predictions_cache.items()):
        c = 2 + m_idx
        p = preds_list[r]
        ov_pred = raw.copy()
        ov_pred[p == 1] = [40, 255, 40]  # Green
        axes[r, c].imshow(ov_pred)
        if r == 0: axes[r, c].set_title(m_name, fontsize=10, fontweight='bold')
        axes[r, c].axis('off')

plt.tight_layout()
os.makedirs("figures", exist_ok=True)
out_fig_path = "figures/sota_comparison_grid.png"
plt.savefig(out_fig_path, dpi=300)
plt.show()
print(f"✅ Saved 300 DPI Comparison Figure: {out_fig_path}")